# Fongbe Challenge — Pipeline Kaggle (setup + fine-tuning w2v-bert-2.0)

1. CSV/scripts depuis GitHub, tars audio depuis le bucket GCS public → extraction → suppression des tars.
2. `train_w2vbert.py` (GitHub) : fine-tuning CTC, checkpoints auto, reprise auto.

Runtime : **GPU T4**.

## 0. Settings

In [ ]:
from pathlib import Path

DATA_DIR = Path('/kaggle/working/fongbe')
OUT_DIR  = Path('/kaggle/working/outputs')
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

REPO_RAW = 'https://raw.githubusercontent.com/Appolinairee/fongbe-challenge/master'
GCS_BASE = 'https://storage.googleapis.com/fongbe'
TAR_PREFIXES = ['audio_train_reliable_']

print('settings ok')

## 1. CSV + scripts depuis GitHub

In [ ]:
import urllib.request
FILES = ['Train_reliable.csv','Train_uncertain.csv','Train_religious.csv','Test.csv',
         'SampleSubmission.csv','syllabify.py','normalize.py','score.py','test_score.py',
         'measure_frugality.py','conventions.md','README.md']
for name in FILES:
    dst = DATA_DIR / name
    if not dst.exists():
        urllib.request.urlretrieve(f'{REPO_RAW}/data/{name}', dst)
        print('✓', name)
print('data files ok:', len(list(DATA_DIR.glob('*.csv'))), 'CSV')

## 2. Télécharger + extraire les archives audio

Chaque tar est téléchargé depuis GCS, extrait dans `DATA_DIR`, puis **supprimé** (quota disque Kaggle).

In [ ]:
import subprocess, shutil, json

TAR_DIR = DATA_DIR / 'tars'
TAR_DIR.mkdir(exist_ok=True)

listing = json.loads(urllib.request.urlopen(
    'https://storage.googleapis.com/storage/v1/b/fongbe/o?maxResults=200&fields=items(name,size)').read())
all_tars = sorted(i['name'] for i in listing['items'] if i['name'].endswith('.tar'))
tars = [t for t in all_tars if any(t.startswith(p) for p in TAR_PREFIXES)]
print(f'{len(tars)} archives sélectionnées')

for name in tars:
    tar_path = TAR_DIR / name
    if not tar_path.exists():
        print(f'download {name}', flush=True)
        subprocess.run(['wget','-q','-O',str(tar_path),f'{GCS_BASE}/{name}'], check=True)
    print(f'extract {name}', flush=True)
    subprocess.run(['tar','-xf',str(tar_path),'-C',str(DATA_DIR)], check=True)
    tar_path.unlink()
    print(f'ok {name}', flush=True)

free = shutil.disk_usage('/kaggle/working').free/1e9
print(f'audio extrait. Disque libre : {free:.1f} GB')

In [ ]:
# Vérification WAV vs CSV
import pandas as pd
for group, csv_name, folder in [('reliable','Train_reliable.csv','audio/train_reliable'),
                                ('religious','Train_religious.csv','audio/train_religious'),
                                ('uncertain','Train_uncertain.csv','audio/train_uncertain'),
                                ('test','Test.csv','audio/test')]:
    sub = DATA_DIR / folder
    n = len(list(sub.glob('*.wav'))) if sub.exists() else 0
    print(f'{group:10s} {n:,} wav')
df = pd.read_csv(DATA_DIR/'Train_reliable.csv')
missing = (~df.audio_path.map(lambda p: (DATA_DIR/p).exists())).sum()
print(f'reliable: {len(df):,} lignes CSV, {missing} wav manquants')

## 3. Entraînement

In [ ]:
# Dépendances + script depuis GitHub
import subprocess, sys, urllib.request
subprocess.run([sys.executable,'-m','pip','install','-q',
                'transformers==5.14.1','soundfile','rapidfuzz==3.14.5'], check=True)
urllib.request.urlretrieve(f'{REPO_RAW}/scripts/train_w2vbert.py',
                           '/kaggle/working/train_w2vbert.py')
print('ok')

In [ ]:
# Entraînement — checkpoints dans /kaggle/working/outputs/w2vbert_kaggle
import subprocess, sys
from pathlib import Path
OUT = Path('/kaggle/working/outputs/w2vbert_kaggle')
result = subprocess.run([
    sys.executable, '/kaggle/working/train_w2vbert.py',
    '--data-dir', '/kaggle/working/fongbe',
    '--output-dir', str(OUT),
    '--max-steps', '4000',
    '--checkpoint-every', '500',
], check=False)
print('exit code:', result.returncode)
print('Modèle + résultats dans:', OUT)